# 🧪 W10-D6 Governance 覆盖图与 Gap 分析

> 配套阅读：同名 `.md`。本 notebook 只用小规模、可重复的模拟来验证核心治理约束。

**实验目标：** 用覆盖矩阵量化 Build/Deploy/Runtime 的治理检查点，并按风险排序缺口。


In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

axes = ["Build", "Deploy", "Runtime"]
controls = ["Prompt custody", "Policy bundle", "Approval", "Audit/Trace", "PII redaction", "Outbound signing"]
# 1=覆盖，0.5=部分覆盖，0=缺失；矩阵让横切缺口显形。
coverage = np.array([
    [1, 0, 0],      # custody
    [0, 1, 1],      # policy
    [0, 1, 1],      # approval
    [.5, .5, 1],    # audit/trace
    [0, 0, .5],     # PII 默认关闭
    [0, 0, 0],      # outbound signing
])
print("每一行 = 一个控制项；每一列 = 生命周期阶段")


In [ ]:
plt.figure(figsize=(8, 4.4))
plt.imshow(coverage, vmin=0, vmax=1, cmap="RdYlGn")
plt.xticks(range(len(axes)), axes); plt.yticks(range(len(controls)), controls)
for i in range(coverage.shape[0]):
    for j in range(coverage.shape[1]):
        label = {0:"缺失", .5:"部分", 1:"覆盖"}[coverage[i, j]]
        plt.text(j, i, label, ha="center", va="center")
plt.colorbar(label="覆盖度"); plt.title("Governance 覆盖图：缺口集中在 Runtime 数据保护与出站边界")
plt.tight_layout(); plt.show()

print("各阶段覆盖均值:", dict(zip(axes, coverage.mean(axis=0).round(2))))


In [ ]:
gaps = [
    {"gap": "PII 默认关闭", "impact": 10, "urgency": 10, "cost": 1},
    {"gap": "出站签名缺失", "impact": 8, "urgency": 7, "cost": 6},
    {"gap": "跨租户测试不足", "impact": 8, "urgency": 6, "cost": 5},
    {"gap": "合规报告缺失", "impact": 5, "urgency": 3, "cost": 5},
]
for g in gaps: g["priority"] = (g["impact"] * g["urgency"]) / g["cost"]
gaps.sort(key=lambda g: g["priority"], reverse=True)
for g in gaps: print(f"P{g['priority']:.1f}  {g['gap']}")

plt.figure(figsize=(7, 3.4))
plt.barh([g["gap"] for g in gaps][::-1], [g["priority"] for g in gaps][::-1], color="#E15759")
plt.xlabel("(影响 × 紧急度) / 修复成本"); plt.title("治理 Gap 的可解释优先级")
plt.tight_layout(); plt.show()
